# Layer 8 Topic Sanity Check

Compare ten short medical prompts with ten short coding prompts using GPT-2 Small and its layer-8 residual-stream SAE. Each prompt contributes its top 20 positive features. This is a small sanity check, not evidence that the method generalizes to other prompts.

A feature is **common** in a group when it appears in at least 5 of its 10 prompts; it is **rare** when it appears in at most 1. A group-enriched feature is common in one group and rare in the other. The verdict calls separation clear only when each group has at least 3 enriched features and the common-feature Jaccard overlap (shared common features divided by the union of common features) is at most 0.25.

In [ ]:
import inspect
import json
import statistics
import sys
from collections import Counter
from importlib.metadata import version
from pathlib import Path
from time import perf_counter

from IPython.display import Markdown, display

cwd = Path.cwd().resolve()
project_root = next(
    (
        path
        for path in (cwd, *cwd.parents)
        if (path / "monitor").is_dir()
        and (path / "eval" / "data" / "sanity_prompts.json").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root from the notebook working directory."
    )
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from monitor.extract import extract_features
from monitor.labels import get_label
from monitor.model import load_model_and_sae

prompt_path = project_root / "eval" / "data" / "sanity_prompts.json"
prompts = json.loads(prompt_path.read_text(encoding="utf-8"))
assert set(prompts) == {"medical", "coding"}
assert all(len(group_prompts) == 10 for group_prompts in prompts.values())
print(f"Project root: {project_root}")
print(f"Prompt counts: { {group: len(items) for group, items in prompts.items()} }")
print(
    "Installed versions: "
    + ", ".join(
        f"{package} {version(package)}" for package in ("sae-lens", "transformer-lens", "torch")
    )
)
print(f"Extraction API: {inspect.signature(extract_features)}")
print(f"Model/SAE loader API: {inspect.signature(load_model_and_sae)}")
print(f"Label API: {inspect.signature(get_label)}")

Project root: /Users/sheddy/Desktop/Feature_monitor
Prompt counts: {'medical': 10, 'coding': 10}
Installed versions: sae-lens 6.54.0, transformer-lens 3.9.0, torch 2.14.1
Extraction API: (prompt: str, k: int = 20) -> list[monitor.extract.FeatureHit]
Model/SAE loader API: () -> tuple[transformer_lens.HookedTransformer.HookedTransformer, sae_lens.saes.sae.SAE]
Label API: (feature_id: 'int | str') -> 'str'


## Load and time extraction
Load the model and SAE before starting the clock. That keeps the per-prompt timings focused on tokenization and feature extraction, rather than a one-time model download and initialization.

In [7]:
model, sae = load_model_and_sae()
print(f"Model device: {model.cfg.device}")
print(f"SAE hook: {sae.cfg.metadata.hook_name}")

feature_hits = {group: [] for group in prompts}
timings_seconds = []
timing_rows = []
for group, group_prompts in prompts.items():
    for prompt_number, prompt in enumerate(group_prompts, start=1):
        started = perf_counter()
        hits = extract_features(prompt, k=20)
        elapsed = perf_counter() - started
        feature_hits[group].append(hits)
        timings_seconds.append(elapsed)
        timing_rows.append((group, prompt_number, elapsed, prompt))

        if len(hits) > 20:
            raise AssertionError("Extraction returned more than 20 features for one prompt.")

print(f"Prompts processed: {len(timings_seconds)}")
print(f"Median extraction time: {statistics.median(timings_seconds):.4f} seconds")
slowest = max(timing_rows, key=lambda row: row[2])
print(
    f"Slowest extraction: {slowest[2]:.4f} seconds ({slowest[0]} prompt {slowest[1]}: {slowest[3]})"
)

timing_table = ["| Group | Prompt | Seconds | Prompt text |", "|---|---:|---:|---|"]
timing_table.extend(
    f"| {group} | {number} | {seconds:.4f} | {prompt} |"
    for group, number, seconds, prompt in timing_rows
)
display(Markdown("\n".join(timing_table)))

Model device: cpu
SAE hook: blocks.8.hook_resid_pre
Prompts processed: 20
Median extraction time: 0.0429 seconds
Slowest extraction: 0.1838 seconds (medical prompt 1: What are common signs of dehydration?)


| Group | Prompt | Seconds | Prompt text |
|---|---:|---:|---|
| medical | 1 | 0.1838 | What are common signs of dehydration? |
| medical | 2 | 0.0440 | Can ibuprofen upset the stomach? |
| medical | 3 | 0.0429 | How is high blood pressure diagnosed? |
| medical | 4 | 0.0427 | What causes seasonal allergies? |
| medical | 5 | 0.0448 | When should a fever be checked by a doctor? |
| medical | 6 | 0.0428 | Can exercise lower blood sugar? |
| medical | 7 | 0.0430 | What is the purpose of a flu vaccine? |
| medical | 8 | 0.0429 | How can I reduce migraine triggers? |
| medical | 9 | 0.0432 | What does a vitamin D test measure? |
| medical | 10 | 0.0433 | Can sleep loss affect heart health? |
| coding | 1 | 0.0429 | How do I reverse a list in Python? |
| coding | 2 | 0.0427 | Why does this JavaScript promise reject? |
| coding | 3 | 0.0426 | How can I sort a table in SQL? |
| coding | 4 | 0.0426 | What does a null pointer error mean? |
| coding | 5 | 0.0427 | How do I read a file in Rust? |
| coding | 6 | 0.0426 | Why is my CSS grid overflowing? |
| coding | 7 | 0.0437 | How can I add a key to a Python dictionary? |
| coding | 8 | 0.0443 | What is the difference between a class and an object? |
| coding | 9 | 0.0437 | How do I resolve a Git merge conflict? |
| coding | 10 | 0.0426 | Why is this regular expression not matching? |

## Features by topic
Count a feature once per prompt when it appears among that prompt's top 20. The table below includes every feature that is common in one topic and rare in the other. Labels come from the existing local cache or Neuronpedia; lookup failures are shown as `unlabelled`.

In [ ]:
frequencies = {
    group: Counter(hit.feature_id for prompt_hits in group_hits for hit in prompt_hits)
    for group, group_hits in feature_hits.items()
}
common_features = {
    group: {feature_id for feature_id, count in group_counts.items() if count >= 5}
    for group, group_counts in frequencies.items()
}
medical_enriched = {
    feature_id
    for feature_id in common_features["medical"]
    if frequencies["coding"][feature_id] <= 1
}
coding_enriched = {
    feature_id
    for feature_id in common_features["coding"]
    if frequencies["medical"][feature_id] <= 1
}

feature_rows = [("Medical-enriched", feature_id) for feature_id in sorted(medical_enriched)] + [
    ("Coding-enriched", feature_id) for feature_id in sorted(coding_enriched)
]
table = [
    "| Group trend | Feature ID | Medical prompts | Coding prompts | Label |",
    "|---|---:|---:|---:|---|",
]
for trend, feature_id in feature_rows:
    label = get_label(feature_id)
    label = label.replace("|", "\\|").replace("\n", " ")
    table.append(
        f"| {trend} | {feature_id} | {frequencies['medical'][feature_id]} / 10 | "
        f"{frequencies['coding'][feature_id]} / 10 | {label} |"
    )
if not feature_rows:
    table.append("| No enriched features | | | | |")
display(Markdown("\n".join(table)))

shared_common = common_features["medical"] & common_features["coding"]
common_union = common_features["medical"] | common_features["coding"]
jaccard_overlap = len(shared_common) / len(common_union) if common_union else 0.0
print(f"Medical common features: {len(common_features['medical'])}")
print(f"Coding common features: {len(common_features['coding'])}")
print(f"Shared common features: {len(shared_common)}")
print(
    f"Common-feature Jaccard overlap: {len(shared_common)} / {len(common_union)} = {jaccard_overlap:.3f}"
)
print(f"Medical-enriched features: {len(medical_enriched)}")
print(f"Coding-enriched features: {len(coding_enriched)}")

| Group trend | Feature ID | Medical prompts | Coding prompts | Label |
|---|---:|---:|---:|---|
| Medical-enriched | 18171 | 5 / 10 | 0 / 10 | medical terms related to inflammation or clinical conditions |
| Medical-enriched | 23838 | 5 / 10 | 1 / 10 | interrogative sentences ending with a question mark |
| Coding-enriched | 10590 | 1 / 10 | 5 / 10 | questions starting with "Can I" or "How can I" |
| Coding-enriched | 18200 | 1 / 10 | 6 / 10 | instances of the article "a" before a noun |
| Coding-enriched | 21629 | 1 / 10 | 5 / 10 | phrases related to instructing or providing guidance on various topics or tasks |
| Coding-enriched | 21981 | 1 / 10 | 5 / 10 | questions starting with "How do you" and "How do we" followed by verbs |

Medical common features: 7
Coding common features: 10
Shared common features: 3
Common-feature Jaccard overlap: 3 / 14 = 0.214
Medical-enriched features: 2
Coding-enriched features: 4


## Verdict
The final cell classifies this run from the counts and overlap shown above: clear requires at least 3 enriched features in each direction and overlap at most 0.25; weak means at least one enriched feature but not clear; not at all means there are no enriched features. It also names alternative layers only for a weak result.

In [9]:
if len(medical_enriched) >= 3 and len(coding_enriched) >= 3 and jaccard_overlap <= 0.25:
    verdict = "clearly"
elif medical_enriched or coding_enriched:
    verdict = "weakly"
else:
    verdict = "not at all"

print(f"Verdict: Layer 8 separates the topics {verdict}.")
print(
    f"Basis: {len(medical_enriched)} medical-enriched features, "
    f"{len(coding_enriched)} coding-enriched features, and "
    f"{len(shared_common)} shared common features out of "
    f"{len(common_union)} common features in the union "
    f"(Jaccard overlap {jaccard_overlap:.3f})."
)
if verdict == "weakly":
    print(
        "Next layers to try: 4 and 11. Before switching, confirm the release "
        "provides the corresponding residual-pre SAE IDs, then update SAE_ID "
        "in monitor/model.py and verify the loaded hook name."
    )
else:
    print(
        "Alternative layers are not proposed by this rule; consider them only if follow-up exploration is warranted."
    )

Verdict: Layer 8 separates the topics weakly.
Basis: 2 medical-enriched features, 4 coding-enriched features, and 3 shared common features out of 14 common features in the union (Jaccard overlap 0.214).
Next layers to try: 4 and 11. Before switching, confirm the release provides the corresponding residual-pre SAE IDs, then update SAE_ID in monitor/model.py and verify the loaded hook name.
